In [ ]:
import kagglehub
import numpy as np
import pandas as pd
import os
import time
import multiprocessing as mp
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score
from itertools import product

In [ ]:
# Descargar el conjunto de datos completo
# Esto devolverá la ruta del directorio local donde se ha descargado el conjunto de datos.
dataset_root_path = kagglehub.dataset_download(
    "meowmeowmeowmeowmeow/gtsrb-german-traffic-sign"
)

# Listar todos los archivos y directorios dentro de la carpeta descargada para asegurar que estén completos
print("\nArchivos y directorios en el conjunto de datos:")
for root, dirs, files in os.walk(dataset_root_path):
    # Imprimir directorios
    for name in dirs:
        print(os.path.join(root, name) + '/')
    # Imprimir archivos
    for name in files:
        print(os.path.join(root, name))

## Reducción de datos
Vamos a reducir el dataset original de a 5k imagenes

In [ ]:
import pandas as pd

train_csv = os.path.join(dataset_root_path, "Train.csv")

df = pd.read_csv(train_csv)

print(df.head())
print(df.shape)

# Pasaremos (por ahora) de 39k imagenes a 5k
df_muestra = df.sample(n=30000, random_state=42)

print(df_muestra.shape)

## Preprocesamiento

In [ ]:
from PIL import Image

X = []
y = []

# iteramos sobre el df para cargar, redimensionar y coleccionar las imagenes y etiquetas
for index, row in df_muestra.iterrows():
    # construimos la ruta completa hacia las imagenes
    image_path = os.path.join(dataset_root_path, row['Path'])

    try:
        # cargar la imagen
        img = Image.open(image_path)
        img = img.resize((96, 96)) # primero fue de 32px-> 64px, ->96px
        # Convertimos la imagen a un arreglo para integrarla a X
        X.append(np.array(img))
        # Integramos ClassId a y
        y.append(row['ClassId'])
    except Exception as e:
        print(f"Error procesando {image_path}: {e}")

# Convertir las listas a arreglos
X = np.array(X)
y = np.array(y)

print(f"Dimensiones de imagenes procesadas (X): {X.shape}")
print(f"Dimensiones de etiquetas (y): {y.shape}")

In [ ]:
import cv2
# aplicaremos filtro gaussiano a cada imagen en X. Inicialmente se aplicó (5,5) pero no dio buenos resultados, reducimos a (3,3)
# El valor 0 indica que la desviación estándar en las direcciones X e Y se calcula a partir del tamaño del kernel
X_smoothed = np.array([cv2.GaussianBlur(img, (3, 3), 0) for img in X])

print(f"dimensiones de imagenes suavizadas(X_smoothed): {X_smoothed.shape}")

### SIFT

Es más comun realizar sift sobre grises perp considerando que el color es importante para este conjunto ya que colores como amarillo, rojos y blancos son de importancia visual va a dejar esta configuracion. Inicialmente sí se hizo sobre grises pero no dio buenos resultados

In [ ]:
import cv2

# inicializar sift
sift = cv2.SIFT_create()

all_descriptors = []

for i, img_gray in enumerate(X_smoothed):
    # Detectar keypoints y descriptores
    keypoints, descriptors = sift.detectAndCompute(img_gray, None)

    if descriptors is not None:
        all_descriptors.append(descriptors)

# Concatenar todos los descriptores en un único array NumPy
# Este array se utilizará para entrenar el vocabulario BOVW

if all_descriptors:
    all_descriptors_np = np.vstack(all_descriptors)
    print(f"Total SIFT descriptores extraídos: {all_descriptors_np.shape}")
else:
    all_descriptors_np = np.array([])
    print("No se extrayeron descriptores SIFT.")

### Creación de vocabulario de bolsa de palabras visuales (BoVW)
Ahora, crearemos un vocabulario visual agrupando los descriptores SIFT mediante K-Means

In [ ]:
from sklearn.cluster import MiniBatchKMeans

# definimos el numero de palabras visuales (clusters)
k = 1000

if all_descriptors_np.shape[0] > 0:
    # Usamos MiniBatchKMeans para mayor eficiencia con grandes conjuntos de datos
    # Es una alternativa más rápida a KMeans
    kmeans = MiniBatchKMeans(n_clusters=k, random_state=42, n_init='auto', verbose=False)
    kmeans.fit(all_descriptors_np)
    visual_vocabulary = kmeans.cluster_centers_
    print(f"Dimensiones del vocabulario visual: {visual_vocabulary.shape}")
else:
    print("No se pudo crear vocabulario,sin descriptores.")
    visual_vocabulary = None

### Generación de vectores de características BoVW
Tras crear el vocabulario, representaremos cada imagen como un histograma de estas palabras visuales. Este histograma cuenta cuántas veces aparece cada palabra visual en una imagen.

In [ ]:
if visual_vocabulary is not None:
    # Inicializar BOWImgDescriptorExtractor
    # Utiliza el detector SIFT y el vocabulario KMeans para generar histogramas
    bow_extractor = cv2.BOWImgDescriptorExtractor(sift, cv2.BFMatcher(cv2.NORM_L2))
    bow_extractor.setVocabulary(visual_vocabulary)

    # Generar vectores de características BoVW para cada imagen
    bovw_features = []
    for img_gray in X_smoothed:
        keypoints = sift.detect(img_gray, None)
        # Calcula el histograma BoVW para la imagen
        if keypoints:
            features = bow_extractor.compute(img_gray, keypoints)
            if features is not None:
                bovw_features.append(features.flatten())
            else:
                # Si el cálculo devuelve None, agregue un vector cero
                bovw_features.append(np.zeros(k))
        else:
            # Si no se detectan puntos clave, agregue un vector cero.
            bovw_features.append(np.zeros(k))

    bovw_features_np = np.array(bovw_features)
    print(f"Características BoVW generadas con dimension: {bovw_features_np.shape}")
else:
    bovw_features_np = None
    print("No se pueden generar las características de BoVW, falta el vocabulario.")

### Análisis de Componentes Principales (PCA)
Finalmente, aplicaremos PCA para reducir la dimensionalidad de los vectores de características BoVW, lo que puede ayudar a mejorar el rendimiento del modelo y reducir el tiempo de cálculo, especialmente si el espacio de características es muy grande

In [ ]:
from sklearn.decomposition import PCA

if bovw_features_np is not None and bovw_features_np.shape[0] > 0:
    # Inicializar el PCA, conservando el 95% de la varianza
    # También se puede especificar un número fijo de componentes, por ejemplo, n_components=50
    pca = PCA(n_components=0.95, random_state=42)

    # Adaptar PCA a las características de BoVW y transformarlas
    bovw_features_pca = pca.fit_transform(bovw_features_np)

    print(f"características BoVW después de PCA: {bovw_features_pca.shape}")
    print(f"número de componentes seleccionados por PCA: {pca.n_components_}")
else:
    bovw_features_pca = None
    print("No se pudo aplicar PCA. Sin características BoVW .")